# 06 · Giving Uncle Tools 🧰

A **tool** is just a normal Python function. The agent reads its name and description, and **decides by itself** when to call it.

```
You: "2 kopi-o peng, how much?"
Agent thinks: I should use calculate_bill
  -> runs YOUR Python function -> gets "$3.00"
Agent: "Two kopi-o peng, $3.00 lah!"
```

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

if not os.path.exists("data"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

MODEL = os.environ.get("OPENAI_MODEL") or "gpt-6-luna"
print("Ready! Using", MODEL)

## 1. The menu (with pandas)

In [ ]:
import pandas as pd

menu = pd.read_csv("data/menu.csv")
menu

In [ ]:
# filter rows: only drinks
menu[menu["category"] == "drink"]

In [ ]:
# look up one price
menu.loc[menu["item"] == "Kopi-O", "price"].iloc[0]

## 2. Turn a function into a tool
The **docstring** (the text in `"""..."""`) is the instruction manual the agent reads. Write it clearly!

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def get_menu(category: str) -> str:
    """Get the kopitiam menu with prices. category: 'drink', 'food', or '' (empty) for everything."""
    rows = menu if category == "" else menu[menu["category"] == category]
    return rows.to_string(index=False)


uncle = Agent(
    name="Kopi Uncle",
    instructions="You are a friendly kopitiam uncle. ALWAYS use get_menu for prices. Never guess. Keep replies short.",
    model=MODEL,
    tools=[get_menu],
)

result = await Runner.run(uncle, "How much is a kopi-o?")
print(result.final_output)

## 3. Peek inside: what did the agent do?

In [ ]:
for item in result.new_items:
    if item.type == "tool_call_item":
        print("🔧 CALLED:", item.raw_item.name, item.raw_item.arguments)
    elif item.type == "tool_call_output_item":
        print("📦 GOT BACK:", str(item.output)[:80], "...")
    elif item.type == "message_output_item":
        print("💬 SAID:", result.final_output)

> 💡 With a real API key you can also see every step at **platform.openai.com → Logs → Traces**.

## 4. A tool with structured input: the bill
`OrderLine` describes **one line of an order**. The agent fills in a *list* of them for us.

In [ ]:
from pydantic import BaseModel

ICED = 0.30   # 'peng' costs 30 cents more


class OrderLine(BaseModel):
    item: str
    quantity: int
    iced: bool


@function_tool
def calculate_bill(order: list[OrderLine]) -> str:
    """Calculate the bill. Use the exact item names from the menu. Set iced=True for 'peng' (iced) drinks."""
    total = 0
    lines = []
    for line in order:
        match = menu[menu["item"].str.lower() == line.item.lower()]
        if match.empty:
            lines.append(f"{line.item}: not on the menu")
            continue
        price = match["price"].iloc[0] + (ICED if line.iced else 0)
        total = total + price * line.quantity
        lines.append(f"{line.quantity} x {line.item}{' Peng' if line.iced else ''} = ${price * line.quantity:.2f}")
    lines.append(f"TOTAL: ${total:.2f}")
    return "\n".join(lines)

## 5. A tool that calls a live API: the weather 🌦️
data.gov.sg gives a free 2-hour forecast for every area in Singapore — no key needed.

In [ ]:
import requests

data = requests.get("https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast").json()
forecasts = data["data"]["items"][0]["forecasts"]
forecasts[:5]

In [ ]:
@function_tool
def get_weather(area: str) -> str:
    """Get the live 2-hour weather forecast for a Singapore area, e.g. 'Bedok', 'Jurong West'. Use 'City' if unsure."""
    data = requests.get("https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast").json()
    for f in data["data"]["items"][0]["forecasts"]:
        if area.lower() in f["area"].lower():
            return f"{f['area']}: {f['forecast']}"
    return f"No area called {area}"

## 6. Put it all together

In [ ]:
uncle = Agent(
    name="Kopi Uncle",
    instructions="""You are a friendly kopitiam uncle in Singapore. Speak light Singlish. Keep replies short.
ALWAYS use get_menu for prices and calculate_bill for totals. Never guess.
'Peng' = iced. If someone asks what to drink, check the weather first.""",
    model=MODEL,
    tools=[get_menu, calculate_bill, get_weather],
)

questions = [
    "2 kopi-o peng and 1 kaya toast set, how much?",
    "I'm in Bedok, what should I drink now?",
]
for q in questions:
    result = await Runner.run(uncle, q)
    tools_used = [i.raw_item.name for i in result.new_items if i.type == "tool_call_item"]
    print("YOU:", q)
    print("TOOLS:", tools_used)
    print("UNCLE:", result.final_output, "\n")

## ✏️ Exercises
1. Add **Teh Halia** (ginger tea, $1.60) to `data/menu.csv`, re-run the menu cell and ask Uncle about it.
2. Write a `get_time()` tool so Uncle says *"Good morning"* or *"Good evening"* correctly (hint: `datetime.now()`).
3. 🌶️ Challenge: a tool `recommend_drink(weather: str)` with your own rules.

In [ ]:
# your code here